# Track 4 — PCRM / MASTERLISTlist Data: dự đoán kết cục mang thai

Huấn luyện và đánh giá ba nhánh baseline của **Track 4** trên bộ **PCRM / MASTERLISTlist** (171 phôi, 146 nhóm
bệnh nhân, nhãn `Outcome` ∈ {0, 1}):

1. **Tabular** — Logistic Regression, Random Forest, LightGBM, XGBoost trên 3 đặc trưng
   Gardner (`Expansion`, `ICM`, `TE`).
2. **Vision-only** — ResNet50 nhận ảnh phôi RGB ghép cùng 3 mặt nạ chuẩn ICM/TE/ZP (6 kênh),
   fine-tune end-to-end.
3. **Multi-modal Late Fusion** — nối feature vector ảnh (2048-d) với feature vector bảng
   (16-d) rồi học chung qua MLP.

Toàn bộ ba nhánh dùng chung một khung đánh giá: `StratifiedGroupKFold` theo cặp bệnh nhân
(`patient_id`), ngưỡng phân loại theo Youden's J, và Group Bootstrap 500 lần cho khoảng tin
cậy 95% của AUC-ROC.

**Dữ liệu cần có**, đặt cùng thư mục với notebook (hoặc sửa đường dẫn trong mục 1):
```
MASTERLISTlistAug30-2017.xlsx
Images/Images/<File Name>.bmp
GT_ICM/GT_ICM/<File Name> ICM_Mask.bmp
GT_TE/GT_TE/<File Name> TE_Mask.bmp
GT_ZP/GT_ZP/<File Name> ZP_Mask.bmp
```

## 1. Cài đặt thư viện

In [ ]:
import os
import re
import warnings

import numpy as np
import pandas as pd
import cv2

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                              confusion_matrix)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import lightgbm as lgb
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tvm

warnings.filterwarnings("ignore")

# ---- Cấu hình đường dẫn: SỬA cho đúng vị trí dữ liệu trên máy bạn ----
DATA_PATH = "MASTERLISTlistAug30-2017.xlsx"   # file Excel lâm sàng + Gardner
IMAGE_BASE_DIR = "./MASTERLIST_data"          # thư mục chứa Images/ , GT_ICM/ , GT_TE/ , GT_ZP/

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Thiết bị tính toán: {DEVICE}")

## 2. Đọc dữ liệu & xử lý đặc trưng Tabular

In [ ]:
df = pd.read_excel(DATA_PATH, sheet_name=0)
print(f"Đã đọc {DATA_PATH}: {df.shape[0]} dòng, {df.shape[1]} cột.")

# Bỏ dòng thiếu File Name / Outcome, chỉ giữ nhãn hợp lệ (0: thất bại, 1: thành công)
df_clean = df.dropna(subset=["File Name", "Outcome"]).copy()
df_clean = df_clean[df_clean["Outcome"].isin([0, 1])].copy()

ID_COL = "File Name"
LABEL_COL = "Outcome"
GROUP_COL = "patient_id"

def extract_patient_id(filename):
    """Tách mã cặp vợ chồng/bệnh nhân từ File Name"""
    match = re.search(r"([A-Za-z]\d{2}-\d+)", str(filename))
    return match.group(1) if match else str(filename)

df_clean[GROUP_COL] = df_clean[ID_COL].apply(extract_patient_id)

# Mã hóa thứ bậc Gardner cho ICM/TE: A -> 3, B -> 2, C -> 1
grade_map = {"A": 3, "B": 2, "C": 1}
df_clean["ICM_num"] = df_clean["ICM"].map(grade_map)
df_clean["TE_num"] = df_clean["TE"].map(grade_map)

FEATURES = ["Expansion", "ICM_num", "TE_num"]

# Điền khuyết bằng median NGAY TRÊN df_clean — để cả nhánh Tabular (qua X) lẫn nhánh
# Vision/Fusion (đọc trực tiếp từ df_clean) dùng chung một giá trị điền khuyết nhất quán.
for col in FEATURES:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

X = df_clean[FEATURES].copy()
y = df_clean[LABEL_COL].astype(int).values
groups = df_clean[GROUP_COL].astype(str).values

print(f"Số phôi hợp lệ (đã bỏ nhãn Unknown): {len(X)}")
print(f"Số cặp bệnh nhân (patient_id): {len(np.unique(groups))}")
print(f"Phân bố nhãn Outcome (0: thất bại, 1: thành công):\n{pd.Series(y).value_counts()}")

## 3. Kiểm tra rò rỉ dữ liệu khi chia tập theo dòng thay vì theo cặp bệnh nhân

Một phôi và các phôi khác của cùng một cặp vợ chồng thường có đặc trưng và kết cục khá giống
nhau. Nếu chia train/test theo dòng (không nhóm theo bệnh nhân), test set có thể "nhìn thấy"
thông tin của cùng cặp bệnh nhân đã xuất hiện ở train, khiến AUC bị thổi phồng. Ô dưới đây so
sánh nhanh AUC khi chia theo dòng (`StratifiedKFold`) với khi chia đúng theo nhóm
(`StratifiedGroupKFold`).

In [ ]:
def cv_auc(splitter, make_model):
    oof = np.zeros(len(y))
    for tr, te in splitter:
        m = make_model().fit(X.iloc[tr], y[tr])
        oof[te] = m.predict_proba(X.iloc[te])[:, 1]
    return roc_auc_score(y, oof)

mk_lgb = lambda: lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                                     min_child_samples=5, random_state=42, verbose=-1)
mk_rf = lambda: RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)

rows = []
for name, mk in [("LightGBM", mk_lgb), ("RandomForest", mk_rf)]:
    a_row = cv_auc(StratifiedKFold(5, shuffle=True, random_state=42).split(X, y), mk)
    a_grp = cv_auc(StratifiedGroupKFold(5, shuffle=True, random_state=42).split(X, y, groups), mk)
    rows.append((name, a_row, a_grp, a_row - a_grp))

leak_tbl = pd.DataFrame(
    rows, columns=["Model", "AUC chia theo DÒNG (sai)", "AUC chia theo CẶP (đúng)", "Chênh lệch"]
).round(3)
print("=== SO SÁNH AUC ĐỂ KIỂM TRA RÒ RỈ DỮ LIỆU ===")
print(leak_tbl.to_string(index=False))

## 4. Hàm tiện ích dùng chung: ngưỡng Youden & Group Bootstrap CI

Hai hàm dưới đây được dùng lại cho **cả ba nhánh** (Tabular, Vision và Fusion) để
đảm bảo mọi nhánh được chấm điểm theo cùng một quy ước.

In [ ]:
def youden_threshold(y_true, y_prob):
    """Ngưỡng phân loại tối ưu theo chỉ số Youden's J (TPR - FPR)."""
    if len(np.unique(y_true)) < 2:
        return 0.5
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    return thr[np.argmax(tpr - fpr)]

def group_bootstrap_ci(y_true, y_prob, groups, n_boot=500, seed=42):
    """Khoảng tin cậy 95% cho AUC-ROC, bootstrap theo nhóm bệnh nhân (không theo từng dòng)."""
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    rng = np.random.default_rng(seed)
    unique_groups = np.unique(groups)
    idx_by_group = {g: np.where(groups == g)[0] for g in unique_groups}
    scores = []
    for _ in range(n_boot):
        sampled_groups = rng.choice(unique_groups, len(unique_groups), replace=True)
        idx = np.concatenate([idx_by_group[g] for g in sampled_groups])
        if len(np.unique(y_true[idx])) == 2:
            scores.append(roc_auc_score(y_true[idx], y_prob[idx]))
    return np.percentile(scores, [2.5, 97.5])

def compute_5_metrics(y_true, y_prob, threshold=None):
    """AUC-ROC, AUC-PR, Precision, Sensitivity, Specificity tại một ngưỡng cho trước
    (hoặc ngưỡng Youden nếu không truyền vào)."""
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    if threshold is None:
        threshold = youden_threshold(y_true, y_prob)
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "AUC-ROC": roc_auc_score(y_true, y_prob),
        "AUC-PR": average_precision_score(y_true, y_prob),
        "Precision": tp / max(tp + fp, 1),
        "Sensitivity": tp / max(tp + fn, 1),
        "Specificity": tn / max(tn + fp, 1),
        "Threshold": threshold,
    }

## 5. Nhánh Tabular: nested `StratifiedGroupKFold` + bảng kết quả

Vòng ngoài (5 fold) chấm điểm trên test; vòng trong (8 fold) tách thêm validation để chọn
ngưỡng Youden, tránh chọn ngưỡng trên chính tập được chấm điểm. Bốn mô hình huấn luyện trên
cùng một cấu hình fold để so sánh công bằng.

In [ ]:
def make_models(seed=42):
    models = {
        "LogReg": lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=3000)),
        "RandomForest": lambda: RandomForestClassifier(
            n_estimators=500, min_samples_leaf=3, n_jobs=-1, random_state=seed),
        "LightGBM": lambda: lgb.LGBMClassifier(
            n_estimators=300, learning_rate=0.03, num_leaves=8, min_child_samples=20,
            subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
            importance_type="gain", random_state=seed, verbose=-1),
    }
    if HAS_XGB:
        models["XGBoost"] = lambda: xgb.XGBClassifier(
            n_estimators=300, learning_rate=0.03, max_depth=3, subsample=0.8,
            colsample_bytree=0.8, eval_metric="auc", random_state=seed, verbosity=0)
    return models

def group_cv(X, y, groups, n_splits=5, seed=42):
    models = make_models(seed)
    n = len(y)
    oof_p = {k: np.zeros(n) for k in models}
    oof_c = {k: np.zeros(n, dtype=int) for k in models}
    fold_auc = {k: [] for k in models}
    importance = []

    outer = StratifiedGroupKFold(n_splits, shuffle=True, random_state=seed)
    for k, (tr, te) in enumerate(outer.split(X, y, groups)):
        inner = StratifiedGroupKFold(8, shuffle=True, random_state=seed + k)
        i_tr, i_va = next(inner.split(X.iloc[tr], y[tr], groups[tr]))
        tr_i, va_i = tr[i_tr], tr[i_va]

        # Bắt buộc: không cặp bệnh nhân nào nằm ở quá một tập train/val/test
        assert not (set(groups[tr_i]) & set(groups[va_i])), "LEAK train/val"
        assert not (set(groups[tr_i]) & set(groups[te])), "LEAK train/test"
        assert not (set(groups[va_i]) & set(groups[te])), "LEAK val/test"

        print(f"fold {k}: train={len(tr_i)} val={len(va_i)} test={len(te)}")
        for name, mk in models.items():
            m = mk().fit(X.iloc[tr_i], y[tr_i])
            thr = youden_threshold(y[va_i], m.predict_proba(X.iloc[va_i])[:, 1])  # ngưỡng chọn trên VAL
            p = m.predict_proba(X.iloc[te])[:, 1]
            oof_p[name][te] = p
            oof_c[name][te] = (p >= thr).astype(int)
            fold_auc[name].append(roc_auc_score(y[te], p))
            if name == "LightGBM":
                importance.append(pd.Series(m.feature_importances_, index=X.columns))
    return oof_p, oof_c, fold_auc, importance

def summarize(y, groups, oof_p, oof_c, fold_auc=None):
    rows = []
    for name in oof_p:
        pred, p = oof_c[name], oof_p[name]
        tp = ((pred == 1) & (y == 1)).sum(); fp_ = ((pred == 1) & (y == 0)).sum()
        tn = ((pred == 0) & (y == 0)).sum(); fn = ((pred == 0) & (y == 1)).sum()
        lo, hi = group_bootstrap_ci(y, p, groups)
        row = {
            "Model": name,
            "AUC-ROC": roc_auc_score(y, p),
            "95% CI": f"[{lo:.3f}, {hi:.3f}]",
            "AUC-PR": average_precision_score(y, p),
            "Precision": tp / max(tp + fp_, 1),
            "Sensitivity": tp / max(tp + fn, 1),
            "Specificity": tn / max(tn + fp_, 1),
        }
        if fold_auc:
            row["AUC/fold"] = f"{np.mean(fold_auc[name]):.3f}±{np.std(fold_auc[name]):.3f}"
        rows.append(row)
    return pd.DataFrame(rows).round(3)

In [ ]:
oof_p, oof_c, fold_auc, importance = group_cv(X, y, groups)
res_tab = summarize(y, groups, oof_p, oof_c, fold_auc)

print(f"\n(AUC-PR ngẫu nhiên = {y.mean():.3f})")
res_tab

## 6. Nhánh Vision-only & Multi-modal Late Fusion

- **Dataset 6 kênh**: ảnh RGB gốc (3 kênh) ghép với 3 mặt nạ chuẩn ICM/TE/ZP (mỗi mặt nạ 1
  kênh xám) → tensor 6×224×224.
- **Vision-only**: ResNet50 có `conv1` được sửa để nhận 6 kênh (3 kênh đầu copy trọng số
  ImageNet gốc, 3 kênh mặt nạ khởi tạo từ trung bình trọng số gốc).
- **Late Fusion**: cùng backbone 6 kênh (bỏ lớp `fc`, giữ lại vector 2048-d) nối với một
  Tabular Encoder nhỏ (3 đặc trưng Gardner → 16-d) rồi qua MLP phân loại — đúng định nghĩa
  "nối feature vector" của Track 4, không phải lấy trung bình xác suất hai mô hình riêng.

In [ ]:
class MASTERLISTEmbryoDataset(Dataset):
    """Đọc ảnh RGB + 3 mặt nạ chuẩn (ICM/TE/ZP) và ghép thành tensor 6 kênh.
    is_multimodal=True thì trả thêm vector 3 đặc trưng Gardner cho nhánh Fusion."""

    def __init__(self, df, base_dir, is_multimodal=False):
        self.df = df.reset_index(drop=True)
        self.is_multimodal = is_multimodal
        self.img_dir = os.path.join(base_dir, "Images", "Images")
        self.icm_dir = os.path.join(base_dir, "GT_ICM", "GT_ICM")
        self.te_dir = os.path.join(base_dir, "GT_TE", "GT_TE")
        self.zp_dir = os.path.join(base_dir, "GT_ZP", "GT_ZP")

    def __len__(self):
        return len(self.df)

    def _load_image_file(self, folder_path, filename, suffix="", is_gray=False):
        base_name = f"{filename}{suffix}"
        for ext in [".BMP", ".bmp", ".png", ".jpg", ".jpeg", ".tif"]:
            full_path = os.path.join(folder_path, base_name + ext)
            if os.path.exists(full_path):
                img = cv2.imread(full_path, cv2.IMREAD_GRAYSCALE if is_gray else cv2.IMREAD_COLOR)
                if img is not None:
                    return img if is_gray else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        # Không tìm thấy file -> trả ảnh đen để pipeline không vỡ (nên kiểm tra log nếu xảy ra nhiều)
        return (np.zeros((224, 224), dtype=np.uint8) if is_gray
                else np.zeros((224, 224, 3), dtype=np.uint8))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fn = str(row["File Name"]).strip()

        rgb = cv2.resize(self._load_image_file(self.img_dir, fn, is_gray=False), (224, 224))
        icm = cv2.resize(self._load_image_file(self.icm_dir, fn, " ICM_Mask", is_gray=True), (224, 224))
        te = cv2.resize(self._load_image_file(self.te_dir, fn, " TE_Mask", is_gray=True), (224, 224))
        zp = cv2.resize(self._load_image_file(self.zp_dir, fn, " ZP_Mask", is_gray=True), (224, 224))

        combined = np.concatenate([rgb, icm[..., None], te[..., None], zp[..., None]], axis=-1)
        tensor_img = torch.tensor(combined.astype(np.float32) / 255.0).permute(2, 0, 1)
        label = torch.tensor(row["Outcome"], dtype=torch.float32)

        if self.is_multimodal:
            tab_feats = torch.tensor(
                [row["Expansion"], row["ICM_num"], row["TE_num"]], dtype=torch.float32)
            return tensor_img, tab_feats, label
        return tensor_img, label


def _expand_conv1_to_6_channels(backbone):
    """Sửa conv1 của ResNet50 để nhận 6 kênh: 3 kênh đầu giữ trọng số ImageNet gốc,
    3 kênh mặt nạ khởi tạo bằng trung bình trọng số gốc (ấm hơn khởi tạo ngẫu nhiên)."""
    old_conv = backbone.conv1
    backbone.conv1 = nn.Conv2d(6, old_conv.out_channels, kernel_size=old_conv.kernel_size,
                                stride=old_conv.stride, padding=old_conv.padding,
                                bias=old_conv.bias)
    with torch.no_grad():
        backbone.conv1.weight[:, :3, :, :] = old_conv.weight
        backbone.conv1.weight[:, 3:, :, :] = old_conv.weight.mean(dim=1, keepdim=True).repeat(1, 3, 1, 1)
    return backbone


class VisionOnlyResNet50(nn.Module):
    """Baseline Vision-only: ResNet50 6 kênh (RGB + 3 mặt nạ), fine-tune end-to-end."""

    def __init__(self, pretrained=True):
        super().__init__()
        weights = tvm.ResNet50_Weights.DEFAULT if pretrained else None
        self.backbone = _expand_conv1_to_6_channels(tvm.resnet50(weights=weights))
        num_ftrs = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(num_ftrs, 1))

    def forward(self, x):
        return self.backbone(x).squeeze(-1)


class LateFusionResNet50(nn.Module):
    """Multi-modal Late Fusion: nối feature vector ảnh (2048-d) với feature vector
    bảng (16-d) rồi học chung qua MLP — cùng backbone 6 kênh với VisionOnlyResNet50."""

    def __init__(self, num_tabular_features=3, pretrained=True):
        super().__init__()
        weights = tvm.ResNet50_Weights.DEFAULT if pretrained else None
        self.backbone = _expand_conv1_to_6_channels(tvm.resnet50(weights=weights))
        num_ftrs = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        self.tab_encoder = nn.Sequential(
            nn.Linear(num_tabular_features, 16), nn.BatchNorm1d(16), nn.ReLU())
        self.classifier = nn.Sequential(
            nn.Linear(num_ftrs + 16, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 1))

    def forward(self, img_x, tab_x):
        v_feat = self.backbone(img_x)
        t_feat = self.tab_encoder(tab_x)
        fusion = torch.cat((v_feat, t_feat), dim=1)
        return self.classifier(fusion).squeeze(-1)

## 7. Huấn luyện 5-fold & đánh giá Vision-only / Fusion

Cùng một cấu hình `StratifiedGroupKFold` (5 fold, theo `patient_id`) được dùng cho cả hai
nhánh để đảm bảo so sánh công bằng, và dùng lại đúng `compute_5_metrics` /
`group_bootstrap_ci` đã định nghĩa ở mục 4 cho nhánh Tabular.

In [ ]:
def train_one_branch(df_clean, base_dir, model_ctor, is_multimodal,
                      n_splits=5, epochs=12, batch_size=16, lr=1e-4, label=""):
    """Huấn luyện + suy luận out-of-fold cho một nhánh (Vision-only hoặc Fusion)."""
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
    groups_ = df_clean["patient_id"].values
    y_all = df_clean["Outcome"].values.astype(int)
    oof_preds = np.zeros(len(df_clean))

    for fold, (tr_idx, val_idx) in enumerate(sgkf.split(df_clean, y_all, groups_)):
        train_ds = MASTERLISTEmbryoDataset(df_clean.iloc[tr_idx], base_dir, is_multimodal)
        val_ds = MASTERLISTEmbryoDataset(df_clean.iloc[val_idx], base_dir, is_multimodal)
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

        model = model_ctor().to(DEVICE)
        criterion = nn.BCEWithLogitsLoss()
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)

        for _ in range(epochs):
            model.train()
            for batch in train_loader:
                if is_multimodal:
                    imgs, tabs, labels = batch
                    imgs, tabs, labels = imgs.to(DEVICE), tabs.to(DEVICE), labels.to(DEVICE)
                    out = model(imgs, tabs)
                else:
                    imgs, labels = batch
                    imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                    out = model(imgs)
                optimizer.zero_grad()
                loss = criterion(out, labels)
                loss.backward()
                optimizer.step()

        model.eval()
        val_preds = []
        with torch.no_grad():
            for batch in val_loader:
                if is_multimodal:
                    imgs, tabs, _ = batch
                    out = model(imgs.to(DEVICE), tabs.to(DEVICE))
                else:
                    imgs, _ = batch
                    out = model(imgs.to(DEVICE))
                val_preds.extend(torch.sigmoid(out).cpu().numpy())

        oof_preds[val_idx] = val_preds
        fold_auc = roc_auc_score(y_all[val_idx], val_preds)
        print(f"  Fold {fold + 1} {label} AUC-ROC: {fold_auc:.3f}")

    return oof_preds


def evaluate_MASTERLIST_branches(df_clean, base_dir, n_splits=5, epochs=12, batch_size=16, lr=1e-4):
    """Huấn luyện + đánh giá cả Vision-only và Fusion, trả về bảng kết quả 5 chỉ số chuẩn."""
    groups_ = df_clean["patient_id"].values
    y_all = df_clean["Outcome"].values.astype(int)
    results = []

    print("=== [1/2] VISION-ONLY (ResNet50, 6 kênh) ===")
    oof_vis = train_one_branch(
        df_clean, base_dir, lambda: VisionOnlyResNet50(pretrained=True), is_multimodal=False,
        n_splits=n_splits, epochs=epochs, batch_size=batch_size, lr=lr, label="Vision")
    m_vis = compute_5_metrics(y_all, oof_vis)
    ci_vis = group_bootstrap_ci(y_all, oof_vis, groups_)
    results.append({
        "Dataset": "MASTERLIST/PCRM Data", "Branch": "Vision-only (ResNet50 6-ch)",
        "AUC-ROC": round(m_vis["AUC-ROC"], 3), "95% CI": f"[{ci_vis[0]:.3f}, {ci_vis[1]:.3f}]",
        "AUC-PR": round(m_vis["AUC-PR"], 3), "Precision": round(m_vis["Precision"], 3),
        "Sensitivity": round(m_vis["Sensitivity"], 3), "Specificity": round(m_vis["Specificity"], 3),
    })

    print("\n=== [2/2] MULTI-MODAL LATE FUSION ===")
    oof_fus = train_one_branch(
        df_clean, base_dir, lambda: LateFusionResNet50(num_tabular_features=3, pretrained=True),
        is_multimodal=True, n_splits=n_splits, epochs=epochs, batch_size=batch_size, lr=lr,
        label="Fusion")
    m_fus = compute_5_metrics(y_all, oof_fus)
    ci_fus = group_bootstrap_ci(y_all, oof_fus, groups_)
    results.append({
        "Dataset": "MASTERLIST/PCRM Data", "Branch": "Multi-modal Late Fusion",
        "AUC-ROC": round(m_fus["AUC-ROC"], 3), "95% CI": f"[{ci_fus[0]:.3f}, {ci_fus[1]:.3f}]",
        "AUC-PR": round(m_fus["AUC-PR"], 3), "Precision": round(m_fus["Precision"], 3),
        "Sensitivity": round(m_fus["Sensitivity"], 3), "Specificity": round(m_fus["Specificity"], 3),
    })

    df_res = pd.DataFrame(results)
    print("\n" + "=" * 70)
    print("BẢNG KẾT QUẢ VISION-ONLY & FUSION — MASTERLIST/PCRM DATA")
    print("=" * 70)
    print(df_res.to_string(index=False))
    return df_res


df_vision_fusion_summary = evaluate_MASTERLIST_branches(df_clean, IMAGE_BASE_DIR, n_splits=5, epochs=12)

## 8. Bảng tổng hợp

Gộp bảng Tabular với bảng Vision/Fusion thành một bảng duy nhất, cùng cấu
trúc cột.

In [ ]:
res_tab_formatted = res_tab.copy()
res_tab_formatted.insert(0, "Dataset", "MASTERLIST/PCRM Data")
res_tab_formatted = res_tab_formatted.rename(columns={"Model": "Branch"})
cols = ["Dataset", "Branch", "AUC-ROC", "95% CI", "AUC-PR", "Precision", "Sensitivity", "Specificity"]

final_summary = pd.concat([res_tab_formatted[cols], df_vision_fusion_summary[cols]], ignore_index=True)

print("=" * 80)
print(" BẢNG TỔNG HỢP TRACK 4 — MASTERLIST/PCRM DATA (Tabular + Vision + Fusion) ")
print("=" * 80)
final_summary